In [1]:
import pandas as pd
import numpy as np
import warnings
import statsmodels.api as sm
warnings.filterwarnings('ignore')

FILE  = 'Данные_pivot.xlsx'
SHEET = 'Лист 1'

df_raw = pd.read_excel(FILE, sheet_name=SHEET)
print(f'Загружено: {df_raw.shape[0]} строк x {df_raw.shape[1]} столбцов')

# ── Cronbach alpha ──
def cronbach(data):
    d = data.dropna()
    if d.shape[0] < 2 or d.shape[1] < 2:
        return np.nan
    k = d.shape[1]
    return (k / (k - 1)) * (1 - d.var(axis=0, ddof=1).sum() / d.sum(axis=1).var(ddof=1))

scales = {
    'PERF_FIN': ['Q8.19_1', 'Q8.19_2', 'Q8.19_3'],
    'CAUS':     ['Q8.17_1', 'Q8.17_2', 'Q8.17_3', 'Q8.17_4', 'Q8.17_5'],
    'ENT_KNOW': ['Q8.18_1', 'Q8.18_2', 'Q8.18_3'],
    'UNI_ENV':  ['Q3.1_1',  'Q3.1_2',  'Q3.1_3',  'Q3.1_4'],
}
print(f"\n{'Шкала':<12} {'alpha':>7} {'N':>6}")
print('-' * 28)
for name, items in scales.items():
    df_raw[name] = df_raw[items].mean(axis=1)
    print(f'{name:<12} {cronbach(df_raw[items]):>7.3f} {df_raw[items].dropna().shape[0]:>6}')

# ── Переменные ──
df = df_raw.copy()

df['PIVOT_COUNT']   = df['Q8.10'].astype(float)
df['ANY_PIVOT']     = (df['PIVOT_COUNT'] > 1).astype(float)
df['PIVOT_DEGREE']  = df['Q8.9'].astype(float)
df['FIRM_AGE']      = df['возраст фирмы'].astype(float)

df['age_log']       = np.log(df['возраст респондентов'].astype(float))
df['firm_age_log']  = np.log(df['FIRM_AGE'])

fs = df['Q8.2'].clip(lower=0).astype(float)
fs[fs > 500] = np.nan
df['FIRM_SIZE_log'] = np.log1p(fs)

df['FEMALE']        = df['Q6.2'].astype(float)
df['PARENTS_ENT']   = df['Q6.4'].astype(float)
df['INTENT_MAIN']   = (df['Q8.3'] == 1).astype(float)
df['SERIAL']        = df['Q8.4'].astype(float)

for s in ['2', '3', '4']:
    df[f'SEC_{s}'] = df[f'Q8.6_{s}'].astype(float)

df['caus_cent'] = df['CAUS'] - df['CAUS'].mean()

print(f'\ncaus_cent: M = {df["caus_cent"].mean():.8f}  (должно быть ≈ 0)')
print(f'age_log:      M = {df["age_log"].mean():.4f}')
print(f'firm_age_log: M = {df["firm_age_log"].mean():.4f}')
print(f'ANY_PIVOT=1:  {(df["ANY_PIVOT"]==1).sum()} ({(df["ANY_PIVOT"]==1).mean():.1%})')

# ── Финальная выборка ──
ALL_VARS = ['PERF_FIN', 'age_log', 'FEMALE', 'PARENTS_ENT', 'INTENT_MAIN',
            'SERIAL', 'firm_age_log', 'FIRM_SIZE_log',
            'SEC_2', 'SEC_3', 'SEC_4',
            'CAUS', 'caus_cent',
            'ANY_PIVOT',
            'PIVOT_COUNT',
            'PIVOT_DEGREE']

md = (df[ALL_VARS]
      .dropna(subset=ALL_VARS)
      .copy()
      .reset_index(drop=True))

print(f'\n{"="*55}')
print(f'Финальная выборка: N = {len(md)}')
print(f'ANY_PIVOT=1: {md["ANY_PIVOT"].mean():.1%} ({int(md["ANY_PIVOT"].sum())} фирм)')
for s in ['2', '3', '4']:
    print(f'  Сектор {s}: {md[f"SEC_{s}"].mean():.1%}  (n={int(md[f"SEC_{s}"].sum())})')

# ── Взаимодействия ──
md['AP0_DEG']      = (1 - md['ANY_PIVOT']) * md['PIVOT_DEGREE']
md['AP1_DEG']      =      md['ANY_PIVOT']  * md['PIVOT_DEGREE']

md['AP0_DEG_CAUS'] = (1 - md['ANY_PIVOT']) * md['PIVOT_DEGREE'] * md['caus_cent']
md['AP1_DEG_CAUS'] =      md['ANY_PIVOT']  * md['PIVOT_DEGREE'] * md['caus_cent']

for s in ['2', '3', '4']:
    AP  = md['ANY_PIVOT']
    SEC = md[f'SEC_{s}']
    DEG = md['PIVOT_DEGREE']
    md[f'AP0_S{s}0_DEG'] = (1-AP) * (1-SEC) * DEG
    md[f'AP0_S{s}1_DEG'] = (1-AP) *    SEC  * DEG
    md[f'AP1_S{s}0_DEG'] =    AP  * (1-SEC) * DEG
    md[f'AP1_S{s}1_DEG'] =    AP  *    SEC  * DEG

# ── OLS ──
def run_ols(xcols):
    missing = [c for c in xcols if c not in md.columns]
    if missing:
        raise ValueError(f'Столбцы не найдены: {missing}')
    X = sm.add_constant(md[xcols], has_constant='add')
    y = md['PERF_FIN']
    return sm.OLS(y, X).fit()

def sig(p):
    if np.isnan(p): return ''
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else '.' if p < .1 else ''

def print_stata_style(label, res):
    n   = int(res.nobs)
    k   = int(res.df_model)
    df_r = int(res.df_resid)
    print(f'\n{"="*72}')
    print(f'  {label}')
    print(f'{"="*72}')
    print(f'  {"Source":<12} {"SS":>14} {"df":>6} {"MS":>14}')
    print(f'  {"─"*50}')
    ss_m = res.ess
    ss_r = res.ssr
    ss_t = res.centered_tss
    print(f'  {"Model":<12} {ss_m:>14.6f} {k:>6} {ss_m/k:>14.6f}')
    print(f'  {"Residual":<12} {ss_r:>14.6f} {df_r:>6} {ss_r/df_r:>14.6f}')
    print(f'  {"─"*50}')
    print(f'  {"Total":<12} {ss_t:>14.6f} {n-1:>6} {ss_t/(n-1):>14.6f}')
    print()
    print(f'  Number of obs = {n:>6}')
    print(f'  F({k},{df_r})       = {res.fvalue:>8.2f}')
    print(f'  Prob > F      = {res.f_pvalue:>8.4f}')
    print(f'  R-squared     = {res.rsquared:>8.4f}')
    print(f'  Adj R-squared = {res.rsquared_adj:>8.4f}')
    print(f'  Root MSE      = {np.sqrt(res.mse_resid):>8.4f}')
    print()
    print(f'  {"PERF_FIN":<32} {"Coef.":>10} {"Std.Err.":>10} {"t":>7} {"P>|t|":>7}  [95% CI]')
    print(f'  {"─"*80}')
    ci = res.conf_int()
    for v in res.params.index:
        b   = res.params[v]
        se  = res.bse[v]
        tv  = res.tvalues[v]
        pv  = res.pvalues[v]
        lo  = ci.loc[v, 0]
        hi  = ci.loc[v, 1]
        vname = v if v != 'const' else '_cons'
        print(f'  {vname:<32} {b:>10.7f} {se:>10.7f} {tv:>7.2f} {pv:>7.3f}  '
              f'[{lo:>9.6f}, {hi:>9.6f}]  {sig(pv)}')
    print(f'  {"─"*80}')

# ── Controls ──
BASE = ['age_log', 'FEMALE', 'PARENTS_ENT', 'INTENT_MAIN', 'SERIAL',
        'firm_age_log', 'FIRM_SIZE_log', 'SEC_2', 'SEC_3', 'SEC_4']

res_m1 = run_ols(BASE + ['CAUS', 'ANY_PIVOT'])
print_stata_style('M1', res_m1)

res_m2 = run_ols(BASE + ['CAUS', 'ANY_PIVOT', 'PIVOT_DEGREE',
                          'AP0_DEG', 'AP1_DEG'])
print_stata_style('M2', res_m2)

res_m3 = run_ols(BASE + [
    'ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
    'AP0_DEG',      'AP1_DEG',
    'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
    'AP0_S40_DEG',  'AP0_S41_DEG',  'AP1_S40_DEG',  'AP1_S41_DEG'
])
print_stata_style('M3',
                  res_m3)

res_m4 = run_ols(BASE + [
    'ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
    'AP0_DEG',      'AP1_DEG',
    'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
    'AP0_S30_DEG',  'AP0_S31_DEG',  'AP1_S30_DEG',  'AP1_S31_DEG'
])
print_stata_style('M4',
                  res_m4)

res_m5 = run_ols(BASE + [
    'ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
    'AP0_DEG',      'AP1_DEG',
    'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
    'AP0_S20_DEG',  'AP0_S21_DEG',  'AP1_S20_DEG',  'AP1_S21_DEG'
])
print_stata_style('M5',
                  res_m5)

# ── Сводная таблица ──
print('\n\n' + '='*90)
print('  СВОДНАЯ ТАБЛИЦА')
print('='*90)

all_models = [
    ('M1', res_m1),
    ('M2', res_m2),
    ('M3 (SEC_4)', res_m3),
    ('M4 (SEC_3)', res_m4),
    ('M5 (SEC_2)', res_m5),
]

KEY_LABELS = {
    'CAUS':         'CAUS',
    'caus_cent':    'caus_cent',
    'ANY_PIVOT':    'ANY_PIVOT',
    'PIVOT_DEGREE': 'PIVOT_DEGREE',
    'AP0_DEG':      'i.AP=0 # c.DEG',
    'AP1_DEG':      'i.AP=1 # c.DEG',
    'AP0_DEG_CAUS': 'i.AP=0 # c.DEG # c.CAUS',
    'AP1_DEG_CAUS': 'i.AP=1 # c.DEG # c.CAUS',
    'AP0_S40_DEG':  'i.AP=0,SEC4=0 # c.DEG',
    'AP0_S41_DEG':  'i.AP=0,SEC4=1 # c.DEG',
    'AP1_S40_DEG':  'i.AP=1,SEC4=0 # c.DEG',
    'AP1_S41_DEG':  'i.AP=1,SEC4=1 # c.DEG',
    'AP0_S30_DEG':  'i.AP=0,SEC3=0 # c.DEG',
    'AP0_S31_DEG':  'i.AP=0,SEC3=1 # c.DEG',
    'AP1_S30_DEG':  'i.AP=1,SEC3=0 # c.DEG',
    'AP1_S31_DEG':  'i.AP=1,SEC3=1 # c.DEG',
    'AP0_S20_DEG':  'i.AP=0,SEC2=0 # c.DEG',
    'AP0_S21_DEG':  'i.AP=0,SEC2=1 # c.DEG',
    'AP1_S20_DEG':  'i.AP=1,SEC2=0 # c.DEG',
    'AP1_S21_DEG':  'i.AP=1,SEC2=1 # c.DEG',
}

def fmt(res, vname):
    params = res.params
    pvals  = res.pvalues
    key = 'const' if vname == '_cons' else vname
    if key not in params.index:
        return f'{"—":>14}'
    b = params[key]; p = pvals[key]
    return f'{b:>+10.4f}{sig(p):>4}'

col_w = 14
hdr = f'{"":38}' + ''.join(f'{m:>{col_w}}' for m, _ in all_models)
print(hdr)
print('─' * (38 + col_w * len(all_models)))

print('  [Контрольные переменные]')
for v in BASE:
    row = f'  {v:<36}' + ''.join(fmt(res, v) for _, res in all_models)
    print(row)

print('  [Предикторы и взаимодействия]')
for v, label in KEY_LABELS.items():
    row = f'  {label:<36}' + ''.join(fmt(res, v) for _, res in all_models)
    if '—' not in row or row.count('—') < len(all_models) - 1:
        print(row)

print(f'  {"_cons":<36}' + ''.join(fmt(res, '_cons') for _, res in all_models))

print('─' * (38 + col_w * len(all_models)))
for stat, vals in [
    ('N',       [int(r.nobs)         for _, r in all_models]),
    ('R²',      [r.rsquared          for _, r in all_models]),
    ('Adj. R²', [r.rsquared_adj      for _, r in all_models]),
    ('F',       [r.fvalue            for _, r in all_models]),
    ('Prob > F',[r.f_pvalue          for _, r in all_models]),
]:
    if stat == 'N':
        row = f'  {stat:<36}' + ''.join(f'{v:>{col_w}.0f}' for v in vals)
    elif stat == 'Prob > F':
        row = f'  {stat:<36}' + ''.join(f'{v:>{col_w}.4f}' for v in vals)
    elif stat == 'F':
        row = f'  {stat:<36}' + ''.join(f'{v:>{col_w}.2f}' for v in vals)
    else:
        row = f'  {stat:<36}' + ''.join(f'{v:>{col_w}.4f}' for v in vals)
    print(row)

print()
print('  Signif.: *** p<.001  ** p<.01  * p<.05  . p<.10')

# ── VIF ──
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(xcols, label=''):
    X_vif = sm.add_constant(md[xcols], has_constant='add')
    vif_df = pd.DataFrame({
        'Variable': X_vif.columns,
        'VIF': [variance_inflation_factor(X_vif.values, i)
                for i in range(X_vif.shape[1])]
    }).query("Variable != 'const'").sort_values('VIF', ascending=False)
    print(f'\n{"="*55}')
    print(f'  VIF — {label}')
    print(f'{"="*55}')
    print(f'  {"Variable":<30} {"VIF":>8}  Статус')
    print(f'  {"─"*46}')
    for _, row in vif_df.iterrows():
        status = '!!! > 10' if row.VIF > 10 else '!  > 5' if row.VIF > 5 else 'OK'
        print(f'  {row.Variable:<30} {row.VIF:>8.3f}  {status}')
    print(f'  {"─"*46}')
    print(f'  Max VIF : {vif_df.VIF.max():.3f}')
    print(f'  Mean VIF: {vif_df.VIF.mean():.3f}')
    return vif_df

vif_m1 = compute_vif(
    BASE + ['CAUS', 'ANY_PIVOT'],
    label='M1: контроли + CAUS + ANY_PIVOT'
)

# ── Сохранение в txt ──
import io, contextlib

lines = []
for label, res in all_models:
    lines.append('='*72)
    lines.append(f'  {label}')
    lines.append('='*72)
    lines.append(f'  N={int(res.nobs)}  R²={res.rsquared:.4f}  '
                 f'Adj.R²={res.rsquared_adj:.4f}  '
                 f'F({int(res.df_model)},{int(res.df_resid)})={res.fvalue:.2f}  '
                 f'Prob>F={res.f_pvalue:.4f}')
    lines.append(f'  {"PERF_FIN":<32} {"Coef.":>12} {"Std.Err.":>10} {"t":>7} {"P>|t|":>7}')
    lines.append('─'*72)
    ci = res.conf_int()
    for v in res.params.index:
        b  = res.params[v]
        se = res.bse[v]
        tv = res.tvalues[v]
        pv = res.pvalues[v]
        vn = '_cons' if v == 'const' else v
        lines.append(f'  {vn:<32} {b:>12.7f} {se:>10.7f} {tv:>7.2f} {pv:>7.3f}  {sig(pv)}')
    lines.append('')

with open('results_pivot.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))



Загружено: 273 строк x 53 столбцов

Шкала          alpha      N
----------------------------
PERF_FIN       0.917    273
CAUS           0.909    267
ENT_KNOW       0.714    271
UNI_ENV        0.878    272

caus_cent: M = -0.00000000  (должно быть ≈ 0)
age_log:      M = 3.1459
firm_age_log: M = 1.3143
ANY_PIVOT=1:  142 (52.0%)

Финальная выборка: N = 245
ANY_PIVOT=1: 53.9% (132 фирм)
  Сектор 2: 11.8%  (n=29)
  Сектор 3: 35.9%  (n=88)
  Сектор 4: 31.0%  (n=76)

  M1
  Source                   SS     df             MS
  ──────────────────────────────────────────────────
  Model            161.106000     12      13.425500
  Residual         429.063615    232       1.849412
  ──────────────────────────────────────────────────
  Total            590.169615    244       2.418728

  Number of obs =    245
  F(12,232)       =     7.26
  Prob > F      =   0.0000
  R-squared     =   0.2730
  Adj R-squared =   0.2354
  Root MSE      =   1.3599

  PERF_FIN                              Coef.   Std.

In [7]:
import pandas as pd
import numpy as np
import warnings
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.discrete.discrete_model import Logit
from scipy.stats import chi2 as chi2_dist
warnings.filterwarnings('ignore')

# ── Вспомогательные функции ──
def sig(p):
    if np.isnan(p): return ''
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else '.' if p < .1 else ''

def print_sep(title='', w=72, ch=''):
    print('\n' + ch*w)
    if title:
        print(f'  {title}')
        print(ch*w)

def fmt_coef(res, vars_to_show, title=''):
    if title:
        print(f'\n  {title}')
    print(f'  {"Variable":<32} {"Coef.":>9} {"SE":>9} {"t/z":>7} {"p":>8}')
    print('  ' + '─'*62)
    for v in vars_to_show:
        key = 'const' if v == '_cons' else v
        idx_list = list(res.params.index)
        if key not in idx_list:
            continue
        b  = res.params[key]
        se = res.bse[key]
        tv = res.tvalues[key]
        pv = res.pvalues[key]
        marker = '  ←' if v in ['AP1_DEG_CAUS','COUNT_DEG_CAUS'] else ''
        print(f'  {v:<32} {b:>9.4f} {se:>9.4f} {tv:>7.3f} {pv:>8.4f}  {sig(pv)}{marker}')

BASE = ['age_log', 'FEMALE', 'PARENTS_ENT', 'INTENT_MAIN', 'SERIAL',
        'firm_age_log', 'FIRM_SIZE_log', 'SEC_2', 'SEC_3', 'SEC_4']

# ── R1 — Ordered Logit ──
print_sep('R1 — ORDERED LOGIT')

md['PERF_FIN_ord'] = md['PERF_FIN'].round().astype(int).clip(1, 7)
print(f'\n  Распределение PERF_FIN_ord:')
vc = md['PERF_FIN_ord'].value_counts().sort_index()
for k, v in vc.items():
    print(f'    {k}: {v} ({v/len(md):.1%})')

def fit_ol(xcols, label):
    av = [c for c in xcols if c in md.columns and md[c].std() > 1e-9]
    res = OrderedModel(md['PERF_FIN_ord'], md[av],
                       distr='logit').fit(method='bfgs', cov_type='HC3',
                                          disp=False, maxiter=1000)
    res._label = label.split(':')[0].strip()
    res._xcols = av
    return res

def print_ol(res):
    print_sep(f'Ordered Logit — {res._label}', ch='─')
    print(f'  N={res.nobs}   Log-Lik={res.llf:.3f}   '
          f'Pseudo R²={res.prsquared:.4f}   '
          f'AIC={res.aic:.1f}   BIC={res.bic:.1f}')
    print(f'\n  {"Variable":<32} {"Coef.":>9} {"SE":>9} {"z":>7} {"p":>8}')
    print('  ' + '─'*62)
    for name in res.params.index:
        if '/' in str(name):
            continue
        b  = res.params[name]
        se = res.bse[name]
        z  = res.tvalues[name]
        p  = res.pvalues[name]
        marker = '' if name == 'AP1_DEG_CAUS' else ''
        print(f'  {name:<32} {b:>9.4f} {se:>9.4f} {z:>7.3f} {p:>8.4f}  {sig(p)}{marker}')
    print('  ' + '─'*62)
    print('  Пороги (thresholds):')
    for name in res.params.index:
        if '/' in str(name):
            print(f'    {name}: {res.params[name]:.4f}  SE={res.bse[name]:.4f}')

ol_m1 = fit_ol(BASE + ['CAUS', 'ANY_PIVOT'], 'M1: контроли + CAUS + ANY_PIVOT')
print_ol(ol_m1)

ol_m2 = fit_ol(BASE + ['CAUS', 'ANY_PIVOT', 'PIVOT_DEGREE', 'AP0_DEG', 'AP1_DEG'],
               'M2: + PIVOT_DEGREE + i.AP#c.DEG')
print_ol(ol_m2)

ol_m3 = fit_ol(BASE + ['ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
                         'AP0_DEG', 'AP1_DEG', 'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
                         'AP0_S40_DEG', 'AP0_S41_DEG', 'AP1_S40_DEG', 'AP1_S41_DEG'],
               'M3: + caus_cent + i.AP#c.DEG#c.caus_cent + i.AP#c.DEG#i.SEC_4')
print_ol(ol_m3)

ol_m4 = fit_ol(BASE + ['ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
                         'AP0_DEG', 'AP1_DEG', 'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
                         'AP0_S30_DEG', 'AP0_S31_DEG', 'AP1_S30_DEG', 'AP1_S31_DEG'],
               'M4: + caus_cent + i.AP#c.DEG#c.caus_cent + i.AP#c.DEG#i.SEC_3')
print_ol(ol_m4)

ol_m5 = fit_ol(BASE + ['ANY_PIVOT', 'PIVOT_DEGREE', 'caus_cent',
                         'AP0_DEG', 'AP1_DEG', 'AP0_DEG_CAUS', 'AP1_DEG_CAUS',
                         'AP0_S20_DEG', 'AP0_S21_DEG', 'AP1_S20_DEG', 'AP1_S21_DEG'],
               'M5: + caus_cent + i.AP#c.DEG#c.caus_cent + i.AP#c.DEG#i.SEC_2')
print_ol(ol_m5)

print_sep('Сравнение OLS vs Ordered Logit — AP1_DEG_CAUS', ch='─')
print(f'  {"Модель":<20} {"OLS b":>9} {"OLS p":>8} {"OL b":>9} {"OL p":>8}')
print('  ' + '─'*55)
ols_list = [(res_m3, ol_m3, 'M3 (SEC_4)'),
            (res_m4, ol_m4, 'M4 (SEC_3)'),
            (res_m5, ol_m5, 'M5 (SEC_2)')]
for ols_r, ol_r, lbl in ols_list:
    ob = ols_r.params.get('AP1_DEG_CAUS', np.nan)
    op = ols_r.pvalues.get('AP1_DEG_CAUS', np.nan)
    lb = ol_r.params.get('AP1_DEG_CAUS', np.nan)
    lp = ol_r.pvalues.get('AP1_DEG_CAUS', np.nan)
    print(f'  {lbl:<20} {ob:>9.4f} {op:>8.4f}{sig(op):>3}  '
          f'{lb:>9.4f} {lp:>8.4f}{sig(lp):>3}')

# ── R2 — Спецификация с PIVOT_COUNT ──
print_sep('R2 — СПЕЦИФИКАЦИЯ С PIVOT_COUNT')

md['PIVOT_COUNT_c'] = md['PIVOT_COUNT'] - md['PIVOT_COUNT'].mean()
print(f'\n  PIVOT_COUNT: M={md["PIVOT_COUNT"].mean():.2f}, SD={md["PIVOT_COUNT"].std():.2f}')

md['COUNT_DEG']      = md['PIVOT_COUNT_c'] * md['PIVOT_DEGREE']
md['COUNT_DEG_CAUS'] = md['PIVOT_COUNT_c'] * md['PIVOT_DEGREE'] * md['caus_cent']

for s in ['2', '3', '4']:
    md[f'COUNT_S{s}_DEG'] = md['PIVOT_COUNT_c'] * md[f'SEC_{s}'] * md['PIVOT_DEGREE']

r_corr = md['PIVOT_COUNT'].corr(md['ANY_PIVOT'])
print(f'  r(COUNT, ANY_PIVOT) = {r_corr:.3f}')

def run_ols_count(xcols):
    X = sm.add_constant(md[xcols], has_constant='add')
    return sm.OLS(md['PERF_FIN'], X).fit()

rc2_m1 = run_ols_count(BASE + ['CAUS', 'PIVOT_COUNT_c'])

rc2_m2 = run_ols_count(BASE + ['CAUS', 'PIVOT_COUNT_c', 'PIVOT_DEGREE', 'COUNT_DEG'])

md['COUNT_S4_DEG']  = md['PIVOT_COUNT_c'] * md['PIVOT_DEGREE'] * md['SEC_4']
md['COUNT_S3_DEG']  = md['PIVOT_COUNT_c'] * md['PIVOT_DEGREE'] * md['SEC_3']
md['COUNT_S2_DEG']  = md['PIVOT_COUNT_c'] * md['PIVOT_DEGREE'] * md['SEC_2']

rc2_m3 = run_ols_count(BASE + ['PIVOT_COUNT_c', 'PIVOT_DEGREE', 'caus_cent',
                                 'COUNT_DEG', 'COUNT_DEG_CAUS', 'COUNT_S4_DEG'])
rc2_m4 = run_ols_count(BASE + ['PIVOT_COUNT_c', 'PIVOT_DEGREE', 'caus_cent',
                                 'COUNT_DEG', 'COUNT_DEG_CAUS', 'COUNT_S3_DEG'])
rc2_m5 = run_ols_count(BASE + ['PIVOT_COUNT_c', 'PIVOT_DEGREE', 'caus_cent',
                                 'COUNT_DEG', 'COUNT_DEG_CAUS', 'COUNT_S2_DEG'])

count_models = [('RC2-M1', rc2_m1), ('RC2-M2', rc2_m2),
                ('RC2-M3 (SEC_4)', rc2_m3), ('RC2-M4 (SEC_3)', rc2_m4),
                ('RC2-M5 (SEC_2)', rc2_m5)]

for lbl, res in count_models:
    print_sep(lbl, ch='─')
    print(f'  N={int(res.nobs)}  R²={res.rsquared:.4f}  Adj.R²={res.rsquared_adj:.4f}  '
          f'F({int(res.df_model)},{int(res.df_resid)})={res.fvalue:.2f}  p={res.f_pvalue:.4f}')
    show = ['CAUS', 'PIVOT_COUNT_c', 'PIVOT_DEGREE',
            'COUNT_DEG', 'COUNT_DEG_CAUS', 'COUNT_S4_DEG', 'COUNT_S3_DEG', 'COUNT_S2_DEG']
    fmt_coef(res, show)

print_sep('Сравнение бинарной (ANY_PIVOT) и непрерывной (PIVOT_COUNT) спецификации', ch='─')
print(f'  {"Модель":<24} {"R² (AP)":>9} {"R² (CN)":>9}  {"COUNT_DEG_CAUS b":>18} {"p":>8}')
print('  ' + '─'*72)
ols_r2s   = [ res_m3.rsquared, res_m4.rsquared, res_m5.rsquared]
count_r2s = [ rc2_m3.rsquared, rc2_m4.rsquared, rc2_m5.rsquared]
labels_c  = ['M3 (SEC_4)','M4 (SEC_3)','M5 (SEC_2)']
for lbl, r_ap, r_cn, cnt_res in zip(labels_c, ols_r2s, count_r2s,
                                     [rc2_m3, rc2_m4, rc2_m5]):
    b_cdc = cnt_res.params.get('COUNT_DEG_CAUS', np.nan)
    p_cdc = cnt_res.pvalues.get('COUNT_DEG_CAUS', np.nan)
    print(f'  {lbl:<24} {r_ap:>9.4f} {r_cn:>9.4f}  {b_cdc:>18.4f} {p_cdc:>8.4f}{sig(p_cdc):>3}')






  R1 — ORDERED LOGIT


  Распределение PERF_FIN_ord:
    1: 14 (5.7%)
    2: 10 (4.1%)
    3: 29 (11.8%)
    4: 68 (27.8%)
    5: 58 (23.7%)
    6: 35 (14.3%)
    7: 31 (12.7%)

────────────────────────────────────────────────────────────────────────
  Ordered Logit — M1
────────────────────────────────────────────────────────────────────────
  N=245   Log-Lik=-394.078   Pseudo R²=0.0979   AIC=824.2   BIC=887.2

  Variable                             Coef.        SE       z        p
  ──────────────────────────────────────────────────────────────
  age_log                            -1.2790    0.8803  -1.453   0.1462  
  FEMALE                             -0.2605    0.2519  -1.034   0.3010  
  PARENTS_ENT                         0.2444    0.2459   0.994   0.3202  
  INTENT_MAIN                        -0.0025    0.3043  -0.008   0.9935  
  SERIAL                             -0.7307    0.5422  -1.348   0.1778  
  firm_age_log                        1.1179    0.6134   1.822   0.0684  .
